# General Tasks – SoSe26 Case Study, Group 43

Group members: Nicolas Alexander Bauer, Roger Alexander Beever, Tobias Fabian Dünnebeil, Baptist Emil Orb, Louise Charlotte Zepter


## 0. Setup and Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # suppress non-critical warnings for a clean, readable notebook

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
np.random.seed(42)

BASE_DATA_DIR = Path("Data") / "IDA SoSe26 - Data"

KOMPONENTE_DIR = BASE_DATA_DIR / "Komponente"         
LOGISTIKVERZUG_DIR = BASE_DATA_DIR / "Logistikverzug"  
EINZELTEIL_DIR = BASE_DATA_DIR / "Einzelteil"         
ZULASSUNG_DIR = BASE_DATA_DIR / "Zulassungen"          
FAHRZEUG_DIR = BASE_DATA_DIR / "Fahrzeug"              

DATA_DIR = KOMPONENTE_DIR
BESTANDTEILE_DIR = FAHRZEUG_DIR


## 1. Logistics and Product Development in the Automobile Industry (8 Points)

### 1.0 Objective

We want to characterize the **logistics delay** of component **K7**: the time between when a produced unit
is issued at the supplier (one day after `Produktionsdatum`) and when it is registered as incoming goods at
the OEM (`Wareneingang`). We will:

1. build a combined "Logistics delay" dataset from `Komponente_K7.csv` and `Logistikverzug_K7.csv`,
2. identify a suitable probability distribution for the delay (goodness-of-fit tests),
3. compute the mean delay in **working days**,
4. visualize the distribution with Plotly, and
5. describe how a decision tree could classify defective (`Fehlerhaft`) components.


### 1.1 Data Import

We use two datasets:

| Dataset | Relevant columns | Role |
|---|---|---|
| `Komponente_K7.csv` | `IDNummer`, `Produktionsdatum`, `Fehlerhaft` | production date of each component |
| `Logistikverzug_K7.csv` | `IDNummer`, `Wareneingang` | date the component arrived at the OEM |

The key variable that connects both datasets is **`IDNummer`**, the unique identifier of a K7 component.


In [ ]:
komponente = pd.read_csv(LOGISTIKVERZUG_DIR / "Komponente_K7.csv", sep=";")
logistik = pd.read_csv(LOGISTIKVERZUG_DIR / "Logistikverzug_K7.csv", sep=",")

print("Komponente_K7.csv:", komponente.shape)
print("Logistikverzug_K7.csv:", logistik.shape)
komponente.head()


In [ ]:
logistik.head()


### 1.2 Data Preparation and Validation

Before merging, we check data types, missing values, duplicates, and whether the key variable
(`IDNummer`) actually matches between both tables — this is essential to trust the later merge.


In [ ]:
for name, df in [("Komponente_K7", komponente), ("Logistikverzug_K7", logistik)]:
    print(f"--- {name} ---")
    print(df.dtypes)
    print("Missing values:\n", df.isna().sum())
    print("Duplicate IDNummer:", df["IDNummer"].duplicated().sum())
    print()


In [ ]:
# Convert date columns from string to datetime
komponente["Produktionsdatum"] = pd.to_datetime(komponente["Produktionsdatum"])
logistik["Wareneingang"] = pd.to_datetime(logistik["Wareneingang"])

# Check how many IDNummer values are shared between both tables (merge feasibility)
common_ids = set(komponente["IDNummer"]).intersection(set(logistik["IDNummer"]))
print(f"IDs in Komponente_K7: {komponente['IDNummer'].nunique()}")
print(f"IDs in Logistikverzug_K7: {logistik['IDNummer'].nunique()}")
print(f"IDs present in BOTH tables: {len(common_ids)}")


All `IDNummer` values are unique in both tables and match completely between the two datasets
(306,490 common IDs), so an **inner join on `IDNummer`** is safe and will not create or drop rows unexpectedly.


### 1.3 Creating the "Logistics delay" Dataset

The task states that produced goods are issued **one day after** the production date. We therefore define:

- `Ausgabedatum` (issue date) = `Produktionsdatum` + 1 day
- `Verzug_Kalendertage` (delay in calendar days) = `Wareneingang` − `Ausgabedatum`


In [ ]:
logistics_delay = komponente[["IDNummer", "Produktionsdatum", "Fehlerhaft"]].merge(
    logistik[["IDNummer", "Wareneingang"]], on="IDNummer", how="inner", validate="one_to_one"
)

logistics_delay["Ausgabedatum"] = logistics_delay["Produktionsdatum"] + pd.Timedelta(days=1)
logistics_delay["Verzug_Kalendertage"] = (
    logistics_delay["Wareneingang"] - logistics_delay["Ausgabedatum"]
).dt.days

print("Rows in merged 'Logistics delay' dataset:", len(logistics_delay))
logistics_delay.head()


In [ ]:
# Validate the merge: no missing values, no negative delays (goods cannot arrive before they are issued)
print("Missing values after merge:\n", logistics_delay.isna().sum())
print("\nNegative delays (data quality issue if > 0):", (logistics_delay["Verzug_Kalendertage"] < 0).sum())
print("\nDescriptive statistics of the delay (calendar days):")
logistics_delay["Verzug_Kalendertage"].describe()


The merge is complete (no missing values, `validate="one_to_one"` confirms a clean 1:1 join) and no
negative delays occur, so the derived dataset is analysis-ready.


### 1a. How is the logistics delay distributed? (2 Points)

**Approach:** We first inspect the shape of the empirical distribution (histogram, skewness, kurtosis).
Since the delay is a **strictly positive, right-skewed count-like variable** (goods can never arrive
"too early", but can be considerably late), typical candidate distributions are the **Normal**, **Log-normal**
and **Gamma** distribution. We fit each candidate to the data with maximum-likelihood estimation
(`scipy.stats.<dist>.fit`) and compare them using:

- **Kolmogorov–Smirnov (KS) test** – compares the empirical CDF to the fitted theoretical CDF,
- **AIC / BIC** – penalized log-likelihood, to rank distributions independently of sample size effects,
- a **Q–Q plot** – visual check of the fit in the tails.

**Important caveat:** with a very large sample (n ≈ 306,000) and a *discrete* underlying variable (delay is
measured in whole days), formal hypothesis tests such as KS or Chi² become extremely "powerful" — they will
reject almost *any* continuous null distribution at p ≈ 0, even a good approximation, simply because of the
sample size. We therefore treat the p-value as informative mainly in *relative* terms (comparing test
statistics/AIC across candidates) rather than as a strict pass/fail criterion, and support the decision with
visual diagnostics.


In [ ]:
data = logistics_delay["Verzug_Kalendertage"].values.astype(float)

print(f"Mean:      {data.mean():.3f}")
print(f"Std.dev.:  {data.std():.3f}")
print(f"Skewness:  {stats.skew(data):.3f}   (0 = symmetric, >0 = right-skewed)")
print(f"Kurtosis:  {stats.kurtosis(data):.3f}")


In [ ]:
def aic_bic(dist, params, data):
    """Compute AIC/BIC for a fitted scipy.stats distribution."""
    log_lik = np.sum(dist.logpdf(data, *params))
    k = len(params)
    n = len(data)
    aic = 2 * k - 2 * log_lik
    bic = k * np.log(n) - 2 * log_lik
    return aic, bic

candidates = ["norm", "lognorm", "gamma"]
fit_results = []

for name in candidates:
    dist = getattr(stats, name)
    params = dist.fit(data)
    D, p_value = stats.kstest(data, name, args=params)
    aic, bic = aic_bic(dist, params, data)
    fit_results.append({"distribution": name, "params": params, "KS_D": D, "KS_p": p_value, "AIC": aic, "BIC": bic})

fit_df = pd.DataFrame(fit_results).sort_values("AIC").reset_index(drop=True)
fit_df


In [ ]:
# Q-Q plots for visual comparison of the two best candidates
fig = make_subplots(rows=1, cols=2, subplot_titles=("Q-Q Plot: Normal", "Q-Q Plot: Log-normal"))

for col, dist_name in zip([1, 2], ["norm", "lognorm"]):
    dist = getattr(stats, dist_name)
    params = fit_df.loc[fit_df["distribution"] == dist_name, "params"].values[0]
    osm, osr = stats.probplot(data, dist=dist, sparams=params, fit=False)  # osm, osr are both 1D arrays
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", marker=dict(size=3), name=dist_name), row=1, col=col)
    line = np.linspace(osm.min(), osm.max(), 2)
    fig.add_trace(go.Scatter(x=line, y=line, mode="lines", line=dict(color="red", dash="dash"), showlegend=False), row=1, col=col)

fig.update_layout(height=450, width=950, title_text="Q-Q Plots: Theoretical vs. Empirical Quantiles")
fig.show()


**Interpretation:** The delay distribution is **right-skewed** (skewness ≈ 0.57), i.e. most components
arrive within a fairly narrow band around the mean, but a longer tail of late deliveries pulls the
distribution to the right. Consistent with this, the **Log-normal** and **Gamma** distributions achieve a
clearly lower AIC/BIC than the Normal distribution, and their Q-Q plot follows the reference line more
closely in the upper tail. All KS tests reject exact equality (p ≈ 0) — expected given the large,
discrete sample discussed above — but based on the **relative fit (AIC/BIC) and the Q-Q diagnostics**, we
conclude that the logistics delay is best approximated by a **Log-normal distribution** (a Gamma
distribution is a close second and would also be a defensible choice). A Normal distribution is not a good
choice because it does not respect the strictly positive support and underestimates the right tail.


### 1b. Mean logistics delay in working days (2 Points)

We now express the delay in **working days** (Monday–Friday only, weekends excluded), using
`numpy.busday_count`, which counts business days between the issue date (`Ausgabedatum`, inclusive) and the
arrival date (`Wareneingang`, exclusive).


In [ ]:
start_dates = logistics_delay["Ausgabedatum"].values.astype("datetime64[D]")
end_dates = logistics_delay["Wareneingang"].values.astype("datetime64[D]")

logistics_delay["Verzug_Arbeitstage"] = np.busday_count(start_dates, end_dates)

mean_working_days = logistics_delay["Verzug_Arbeitstage"].mean()
median_working_days = logistics_delay["Verzug_Arbeitstage"].median()

print(f"Mean logistics delay:   {mean_working_days:.2f} working days")
print(f"Median logistics delay: {median_working_days:.2f} working days")
print(f"Std. dev.:              {logistics_delay['Verzug_Arbeitstage'].std():.2f} working days")


**Interpretation:** On average, a K7 component takes about **4.3 working days** to travel from the
supplier to the OEM's incoming-goods department (compared to ≈ 6.1 *calendar* days — the difference is
explained by the weekend days that are excluded from the working-day count).

**Alternatives to the arithmetic mean:**
- The **median** is more robust to the right-skewed tail of very late deliveries and may better represent
  the "typical" delay experienced by most shipments.
- A **trimmed mean** (e.g., excluding the top/bottom 1–5 %) would reduce the influence of extreme outliers
  while still using most of the data.
- Reporting the mean **together with a percentile-based service level** (e.g., "90 % of shipments arrive
  within X working days") is often more actionable for logistics planning than a single average, because it
  directly informs buffer/safety-stock decisions.
- Public holidays are not accounted for by `numpy.busday_count` by default; including a holiday calendar
  (`numpy.busdaycalendar`) would make the working-day estimate more precise.


### 1c. Visualization: Histogram and Density Function (2 Points)

We visualize the calendar-day delay distribution with **Plotly**, overlaying the empirical histogram with
the fitted Log-normal density curve identified in 1a.

**Bin size selection:** Since the delay is an *integer-valued* variable (whole days), the most interpretable
choice is **one bin per integer day** (bin width = 1), so that each bar directly represents "share of
shipments with exactly k days of delay" rather than mixing several day-values into one bar. This also avoids
arbitrary binning rules (e.g., Freedman–Diaconis, Sturges) producing bin edges that fall *between* integers,
which would be misleading for discrete data.


In [ ]:
best_dist_name = fit_df.iloc[0]["distribution"]  # best fit by AIC (lognorm)
best_params = fit_df.iloc[0]["params"]
best_dist = getattr(stats, best_dist_name)

x_range = np.linspace(data.min(), data.max(), 300)
pdf_values = best_dist.pdf(x_range, *best_params)

bin_edges = np.arange(data.min() - 0.5, data.max() + 1.5, 1)  # one bin per integer day

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=data, xbins=dict(start=bin_edges[0], end=bin_edges[-1], size=1),
    histnorm="probability density", name="Empirical delay", marker_color="#4C72B0", opacity=0.75
))
fig.add_trace(go.Scatter(
    x=x_range, y=pdf_values, mode="lines", name=f"Fitted {best_dist_name} density",
    line=dict(color="#C44E52", width=3)
))

fig.update_layout(
    title="Distribution of the K7 Logistics Delay (Calendar Days)",
    xaxis_title="Delay [calendar days]",
    yaxis_title="Density",
    bargap=0.05,
    template="plotly_white",
    width=850, height=500
)
fig.show()


### 1d. Decision Tree for Classifying Defective Components (2 Points)

**Task:** describe the process of building a decision tree that classifies whether a K7 component is
defective (`Fehlerhaft`) or not.

**Process:**

1. **Define features and target.** The target is the binary column `Fehlerhaft` (0 = OK, 1 = defective).
   Candidate features available in `Komponente_K7.csv` are `Herstellernummer`, `Werksnummer`, and
   time-derived features from `Produktionsdatum` (e.g., production year/month/weekday) — the reasoning being
   that certain plants, manufacturers, or production periods may be systematically associated with quality
   issues.
2. **Exploratory check of class balance.** Before modelling, inspect how many defective vs. non-defective
   parts exist — this determines whether special handling (class weights, resampling) is required.
3. **Feature engineering.** Extract numeric/categorical features from the date (year, month, day-of-week),
   and encode categorical variables (`Herstellernummer`, `Werksnummer`) if needed (they are already numeric
   codes here).
4. **Train/test split.** Split the data (e.g. 70/30 or 80/20) using **stratified sampling** on `Fehlerhaft`
   so that both sets preserve the (likely very low) share of defective parts.
5. **Model training.** Fit a `sklearn.tree.DecisionTreeClassifier`, using `class_weight="balanced"` to
   compensate for class imbalance, and constrain complexity (`max_depth`, `min_samples_leaf`) to avoid
   overfitting on a rare-event target.
6. **Visualization & interpretation.** Plot the tree (`plot_tree`) and feature importances to understand
   which splits (e.g., specific plant or production month) are associated with higher defect risk.
7. **Evaluation.** Because the target is rare, accuracy is a misleading metric — use **precision, recall,
   F1-score, and the confusion matrix** (or ROC-AUC) instead, focusing on recall for the defective class if
   the business goal is to catch as many defective parts as possible.

Below is an illustrative implementation with the available features:


In [ ]:
model_data = komponente.copy()
model_data["Produktionsjahr"] = model_data["Produktionsdatum"].dt.year
model_data["Produktionsmonat"] = model_data["Produktionsdatum"].dt.month
model_data["Produktionswochentag"] = model_data["Produktionsdatum"].dt.dayofweek

print("Class balance (Fehlerhaft):")
print(model_data["Fehlerhaft"].value_counts(normalize=True))


In [ ]:
features = ["Herstellernummer", "Werksnummer", "Produktionsjahr", "Produktionsmonat", "Produktionswochentag"]
X = model_data[features]
y = model_data["Fehlerhaft"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree_clf = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=50, class_weight="balanced", random_state=42
)
tree_clf.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(tree_clf, feature_names=features, class_names=["OK", "Fehlerhaft"], filled=True, fontsize=8)
plt.title("Decision Tree: Classifying Defective K7 Components")
plt.show()


In [ ]:
importances = pd.Series(tree_clf.feature_importances_, index=features).sort_values(ascending=False)

fig = px.bar(
    importances, orientation="h", labels={"index": "Feature", "value": "Importance"},
    title="Feature Importance – Decision Tree for 'Fehlerhaft'", template="plotly_white"
)
fig.update_layout(showlegend=False, width=750, height=400)
fig.show()


**Note on this specific dataset:** in this sample, only 6 out of 306,490 components are marked as
defective (≈ 0.002 %). With such extreme class imbalance and only a handful of positive cases, *any*
classifier — however well built — will have very limited statistical power, and results should be
interpreted with caution (the tree above is illustrative of the *process*, not a reliable production model).
In practice, more positive examples and richer features (e.g., material batch, supplier quality scores)
would be needed for a robust classifier.


## 2. Data Storage in Separate Files (2 Points)

**Why store data in separate files instead of one large table?**

1. **Reduced redundancy** – shared attributes (e.g., `Herstellernummer`, `Werksnummer`) are stored once per
   entity instead of being repeated in every row of a giant combined table, saving storage and avoiding
   update anomalies.
2. **Data integrity & consistency** – if a value (e.g., a plant's address) needs to change, it only has to be
   updated in one place, instead of in every row of a flattened mega-table, preventing inconsistent copies.
3. **Modularity and maintainability** – each file/table represents one entity (components, logistics events,
   registrations, etc.), which makes the schema easier to understand, extend, and maintain than a single wide
   table mixing unrelated concepts.
4. **Performance and scalability** – smaller, well-indexed tables are faster to query, join selectively, and
   load into memory only when needed, whereas one huge denormalized table wastes memory and I/O on columns
   that are irrelevant to a given analysis.
5. **Referential integrity** – keys (e.g., `IDNummer`) can enforce valid relationships between tables (e.g.,
   a `Bestandteile_Komponente_K7` record must reference an existing component), which is not possible within
   a single flat file.

**Structure name:** The provided tables (separate entity tables connected via key/ID columns, e.g.
`IDNummer`, `ID_T34`, …) follow the structure of a **relational database** (normalized, entity–relationship
model). The specific pattern here — a central "bill of materials"-style bridge table (`Bestandteile_...`)
linking a component to its constituent parts — is a classic **relational (normalized) schema** used to model
hierarchical part structures.


## 3. Parts T16 in Registered Vehicles (3 Points)

### 3.1 Data Import and First Inspection

We use `Einzelteil_T16.txt` (production/failure data of individual T16 parts) and
`Zulassungen_alle_Fahrzeuge.csv` (vehicle registrations). To answer *"how many T16 parts ended up in vehicles
registered in Adelshofen"*, we would need a **key table** that links a T16 serial number to the vehicle
(`IDNummer` in `Zulassungen_alle_Fahrzeuge`) it was installed in — analogous to `Bestandteile_Komponente_K7.csv`,
which linked component K7 to parts T34/T35/T38/T39/T40.

**`Einzelteil_T16.txt` itself does not contain such a link** (it only has production/failure attributes of the
T16 part itself, no vehicle or Karosserie ID), so before that bridge table is available we cannot finish the
count. What we *can* do already is clean and prepare the T16 data, so the join is a single step once the
missing file arrives.


### 3.2 Data Preparation: Fixing the `Einzelteil_T16.txt` File Structure

A first look at the raw file reveals a data-quality problem: the file contains **no newline characters at
all** — rows are separated by tabs (`\t`) and fields within a row by `" | | "`. In addition, the header lists
three groups of the same 7 columns with suffixes `.x`, `.y`, and no suffix (as pandas/R produce when merging
tables with overlapping column names). Inspecting the data shows that **for every row exactly one of the
three groups is populated and the other two are entirely `NA`** — i.e., three separate part-batches were
accidentally combined column-wise (`cbind`) instead of row-wise (`rbind` / `concat`).

We fix this by:
1. Parsing the file with the correct delimiters,
2. splitting it into its three column-blocks,
3. renaming each block to a common schema, and
4. stacking (`concat`) them into one clean, long table of individual T16 parts.


In [ ]:
import io

with open(EINZELTEIL_DIR / "Einzelteil_T16.txt", "r", encoding="utf-8", errors="replace") as f:
    raw_content = f.read()

print("Newline characters in file:", raw_content.count("\n"))
print("Tab-separated records:", raw_content.count("\t"))


In [ ]:
# Replace the non-standard delimiters with a proper CSV structure:
# " | | " -> field separator ";", tab "\t" -> row separator "\n"
cleaned_content = raw_content.replace(" | | ", ";").replace("\t", "\n")

# The header only lists 22 names but each data row has 23 fields (an unnamed row-index
# column precedes the three renamed .x / .y / unsuffixed column groups) -> supply names explicitly
base_cols = ["ID_T16", "Produktionsdatum", "Herstellernummer", "Werksnummer",
             "Fehlerhaft", "Fehlerhaft_Datum", "Fehlerhaft_Fahrleistung"]
col_names = ["_rowname", "_rowidx"] + [c + "_x" for c in base_cols] + [c + "_y" for c in base_cols] + base_cols

einzelteil_raw = pd.read_csv(
    io.StringIO(cleaned_content), sep=";", header=None, names=col_names,
    skiprows=1, na_values="NA", quotechar='"', low_memory=False
)
print("Raw shape (one row per original record):", einzelteil_raw.shape)

# Stack the three column-blocks (.x / .y / unsuffixed) into one long table
blocks = []
for suffix in ["_x", "_y", ""]:
    block = einzelteil_raw[[c + suffix for c in base_cols]].copy()
    block.columns = base_cols
    blocks.append(block)

einzelteil_t16 = pd.concat(blocks, ignore_index=True).dropna(subset=["ID_T16"]).reset_index(drop=True)
einzelteil_t16["Produktionsdatum"] = pd.to_datetime(einzelteil_t16["Produktionsdatum"])

print("Cleaned 'Einzelteil_T16' shape:", einzelteil_t16.shape)
print("Duplicate ID_T16 after cleaning:", einzelteil_t16["ID_T16"].duplicated().sum())
einzelteil_t16.head()


The cleaned table contains **818,844 unique T16 parts** (no duplicates), each with its own production and
failure attributes — this confirms the three original blocks really were disjoint batches that had been
column-bound by mistake.


### 3.3 Registrations in Adelshofen

We can already inspect the registration side. A quick check of the `Gemeinden` column reveals a **second
data-quality issue**: the municipality appears both as `"ADELSHOFEN"` and `"ADELSHOFEN1"` (132 registrations
each) — most likely a naming collision that was resolved by appending `"1"` to one of two distinct places
that happen to share the name "Adelshofen" (Germany has more than one municipality called Adelshofen), or a
duplicate-key artifact from combining regional files. We report both cases separately below and flag this
for clarification rather than silently merging them.


In [ ]:
zulassungen = pd.read_csv(ZULASSUNG_DIR / "Zulassungen_alle_Fahrzeuge.csv", sep=";")
zulassungen["Zulassung"] = pd.to_datetime(zulassungen["Zulassung"])

adelshofen_variants = zulassungen.loc[
    zulassungen["Gemeinden"].str.contains("ADELSHOFEN", case=False, na=False), "Gemeinden"
].value_counts()
print(adelshofen_variants)

vehicles_adelshofen = zulassungen[zulassungen["Gemeinden"].isin(["ADELSHOFEN", "ADELSHOFEN1"])]
print(f"\nVehicles registered in Adelshofen (both spellings): {len(vehicles_adelshofen)}")


### 3.4 Linking Vehicles to Their Major Components

You have now provided the **vehicle bill-of-materials (BOM) files** — one per OEM/vehicle type
(`Bestandteile_Fahrzeuge_OEM1_Typ11.csv`, `..._Typ12.csv`, `..._OEM2_Typ21.csv`, `..._Typ22.csv`). Each links
a vehicle (`ID_Fahrzeug`, matching `IDNummer` in `Zulassungen_alle_Fahrzeuge`) to its four major components:
`ID_Karosserie` (body), `ID_Schaltung` (transmission), `ID_Sitze` (seats), and `ID_Motor` (engine).

Interestingly, this also reveals that the **"Komponente K7" we analyzed in Task 1 is exactly the Karosserie
(body) used in OEM2 Typ22 vehicles** — its `IDNummer` values (e.g. `K7-114-1142-1`) match `ID_Karosserie` in
`Bestandteile_Fahrzeuge_OEM2_Typ22.csv` one-to-one. Similarly, K4/K5/K6 are the Karosserie components of
OEM1 Typ11, OEM1 Typ12, and OEM2 Typ21, respectively.

We combine all four BOM files into one master table and merge it with the registration data.


In [ ]:
bom_files = {
    "OEM1_Typ11": "Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "OEM1_Typ12": "Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    "OEM2_Typ21": "Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
    "OEM2_Typ22": "Bestandteile_Fahrzeuge_OEM2_Typ22.csv",
}

bom_parts = []
for label, filename in bom_files.items():
    part = pd.read_csv(BESTANDTEILE_DIR / filename, sep=";")
    part = part.drop(columns=[c for c in part.columns if c.startswith("Unnamed") or c == "X1"])
    part["OEM_Typ"] = label
    bom_parts.append(part)

fahrzeuge_bom = pd.concat(bom_parts, ignore_index=True)
print("Combined vehicle BOM table:", fahrzeuge_bom.shape)
print("Duplicate ID_Fahrzeug:", fahrzeuge_bom["ID_Fahrzeug"].duplicated().sum())
fahrzeuge_bom.head()


In [ ]:
# Validate: every vehicle in the BOM table should have exactly one registration, and vice versa
merge_check = fahrzeuge_bom.merge(zulassungen, left_on="ID_Fahrzeug", right_on="IDNummer", how="outer", indicator=True)
print(merge_check["_merge"].value_counts())


The merge indicator confirms a **perfect 1:1 match** (`both` for all 3,204,104 rows) — every registered
vehicle has exactly one BOM record and vice versa. We can now find the BOM (and thus the components) of the
vehicles registered in Adelshofen:


In [ ]:
bom_adelshofen = fahrzeuge_bom.merge(vehicles_adelshofen, left_on="ID_Fahrzeug", right_on="IDNummer")
print(f"Vehicles registered in Adelshofen with known component IDs: {len(bom_adelshofen)}")
bom_adelshofen[["ID_Fahrzeug", "ID_Karosserie", "ID_Schaltung", "ID_Sitze", "ID_Motor", "Gemeinden"]].head()


### 3.5 Linking T16 to the Vehicle BOM

`Bestandteile_Komponente_K2LE2.csv` and `Bestandteile_Komponente_K2ST2.csv` reveal that `T16` is a sub-part
of **two** of the four seat sub-types: `K2LE2` (together with `T19`, `T20`) and `K2ST2` (together with `T17`,
`T18`). The other two seat sub-types, `K2ST1` (built from `T11`, `T12`, `T13`) and `K2LE1` (built from `T11`,
`T14`, `T15`), do **not** contain a `T16` part at all — confirmed by checking their bridge files directly.

We combine both `T16` bridge tables (renaming their component-ID column to a common `ID_Sitze` so they can
be matched against the vehicle BOM) and merge with the Adelshofen vehicles.


In [ ]:
t16_le2 = pd.read_csv(DATA_DIR / "Bestandteile_Komponente_K2LE2.csv", sep=";")[["ID_T16", "ID_K2LE2"]]
t16_le2 = t16_le2.rename(columns={"ID_K2LE2": "ID_Sitze"})

t16_st2 = pd.read_csv(DATA_DIR / "Bestandteile_Komponente_K2ST2.csv", sep=";")[["ID_T16", "ID_K2ST2"]]
t16_st2 = t16_st2.rename(columns={"ID_K2ST2": "ID_Sitze"})

t16_bridge = pd.concat([t16_le2, t16_st2], ignore_index=True)

print("Combined T16 bridge (K2LE2 + K2ST2):", t16_bridge.shape)
print("Unique T16 parts:", t16_bridge["ID_T16"].nunique())
print("Unique seat components covered:", t16_bridge["ID_Sitze"].nunique())

# Sanity check: this should match the 818,844 individual T16 parts found earlier
# in the cleaned Einzelteil_T16 table (Section 3.2) -- confirms K2ST1/K2LE1 correctly contain no T16.
print("\nMatches cleaned Einzelteil_T16 count (818,844)?",
      t16_bridge["ID_T16"].nunique() == einzelteil_t16["ID_T16"].nunique())

print("\nSeat sub-type distribution among Adelshofen vehicles:")
print(bom_adelshofen["ID_Sitze"].str.split("-").str[0].value_counts())


In [ ]:
t16_adelshofen = t16_bridge.merge(bom_adelshofen, on="ID_Sitze")

n_t16_adelshofen = t16_adelshofen["ID_T16"].nunique()
print(f"T16 parts installed in vehicles registered in Adelshofen: {n_t16_adelshofen}")
print(t16_adelshofen["ID_Sitze"].str.split("-").str[0].value_counts())
t16_adelshofen[["ID_T16", "ID_Sitze", "ID_Fahrzeug", "Gemeinden"]].head()


### Result

**96 T16 parts** ended up in vehicles registered in Adelshofen: 71 installed via `K2ST2`-type seats and 25
via `K2LE2`-type seats, out of the 264 Adelshofen-registered vehicles in total. The other two seat sub-types
present in Adelshofen (`K2ST1`: 139 vehicles, `K2LE1`: 29 vehicles) do not use a `T16` part, so they do not
contribute to the count. The consistency check against the independently cleaned `Einzelteil_T16` table
(818,844 parts in both) confirms the two bridge tables jointly cover the *entire* T16 population, so no
further seat sub-types are missing.


## 4. Attributes of the Registration Table (2 Points)

We inspect the data types of `Zulassungen_alle_Fahrzeuge` (already loaded above as `zulassungen`) and classify
each attribute both by its **technical (pandas) data type** and its **statistical measurement scale**.


In [ ]:
print(zulassungen.dtypes)
print()
print("Missing values:\n", zulassungen.isna().sum())
print()
print(f"Unique IDNummer: {zulassungen['IDNummer'].nunique()} (of {len(zulassungen)} rows)")
print(f"Unique Gemeinden: {zulassungen['Gemeinden'].nunique()}")
print(f"Zulassung date range: {zulassungen['Zulassung'].min().date()} to {zulassungen['Zulassung'].max().date()}")


| Attribute | Pandas dtype | Statistical scale | Description |
|---|---|---|---|
| `Unnamed: 0` | `int64` | Discrete numeric (technical) | Row index carried over from the original export; not a substantive attribute of the vehicle. |
| `IDNummer` | `object` (string) | **Nominal** | Unique identifier of a vehicle (e.g. `"11-1-11-1"`). Values are unordered labels; arithmetic on them is meaningless. |
| `Gemeinden` | `object` (string) | **Nominal (categorical)** | Name of the municipality where the vehicle is registered. Unordered categories; contains 5,764 distinct values (including a data-quality duplicate, `"ADELSHOFEN"` vs. `"ADELSHOFEN1"`, see Section 3.3). |
| `Zulassung` | `object` → converted to `datetime64` | **Interval scale (date/time)** | Registration date. Differences between dates are meaningful (e.g. "3 days apart"), but there is no true zero, so ratios are not meaningful. |

**Characteristics of the data types:**
- **Nominal** variables (`IDNummer`, `Gemeinden`) represent unordered categories/labels — the only valid
  operations are equality checks and counting frequencies; no meaningful order or arithmetic exists.
- **Interval-scaled date/time** variables (`Zulassung`) support ordering and computing differences (durations),
  but ratios are not meaningful (e.g. "2018 is not twice 2009").
- Storing `IDNummer` and `Gemeinden` as strings (`object`/pandas `str`) rather than numeric types correctly
  reflects that they are identifiers/labels, not quantities to be averaged or summed.


## 5. Linear Model for Mileage (5 Points)

### 5.1 Objective and Data Import

We build a linear model that relates the mileage at failure (`Fehlerhaft_Fahrleistung`, in km) to suitable
explanatory variables from `Fahrzeuge_OEM1_Typ11_Fehleranalyse`, in order to derive recommendations for OEM1.


In [ ]:
fehleranalyse = pd.read_csv(FAHRZEUG_DIR / "Fahrzeuge_OEM1_Typ11_Fehleranalyse.csv", sep=",")
fehleranalyse["Fehlerhaft_Datum"] = pd.to_datetime(fehleranalyse["Fehlerhaft_Datum"])

print(fehleranalyse.shape)
print(fehleranalyse.dtypes)
print(fehleranalyse.isna().sum())
fehleranalyse.head()


### 5.2 Exploratory Data Analysis

The table provides, per vehicle: `days` (days between reference event and failure), `fuel` (fuel consumption
in l/100km), and `engine` (engine-size class: small / medium / large) — all plausible drivers of how many
kilometers a vehicle accumulates before a defect occurs. We check candidate predictors before modelling.


In [ ]:
print("Correlation with target (Fehlerhaft_Fahrleistung):")
print(fehleranalyse[["Fehlerhaft_Fahrleistung", "days", "fuel"]].corr()["Fehlerhaft_Fahrleistung"])
print()
print("Mean mileage and fuel consumption by engine class:")
print(fehleranalyse.groupby("engine")[["Fehlerhaft_Fahrleistung", "fuel"]].mean().sort_values("Fehlerhaft_Fahrleistung"))


In [ ]:
fig = px.scatter(
    fehleranalyse.sample(5000, random_state=42), x="fuel", y="Fehlerhaft_Fahrleistung", color="engine",
    labels={"fuel": "Fuel consumption [l/100km]", "Fehlerhaft_Fahrleistung": "Mileage at failure [km]"},
    title="Mileage at Failure vs. Fuel Consumption, by Engine Class (5,000-point sample)",
    template="plotly_white", opacity=0.5
)
fig.update_layout(width=800, height=500)
fig.show()


**Observations:**
- `fuel` (fuel consumption) is **strongly positively correlated** with mileage at failure (r ≈ 0.69) — vehicles
  with higher consumption accumulate substantially more kilometers before a defect occurs.
- `days` is **essentially uncorrelated** with mileage (r ≈ 0.002) — the time since the reference date alone
  does not explain how many kilometers a vehicle has driven.
- `engine` class and `fuel` are strongly related (large-engine vehicles have far higher average consumption),
  so we should check for **multicollinearity** before interpreting individual coefficients.


### 5.3 Model Specification and Estimation

We fit an OLS (ordinary least squares) linear regression:

$$\text{Fehlerhaft\_Fahrleistung} = \beta_0 + \beta_1 \cdot \text{days} + \beta_2 \cdot \text{fuel} + \beta_3 \cdot \text{engine}_{medium} + \beta_4 \cdot \text{engine}_{large} + \varepsilon$$

`engine` is a categorical variable, so it is one-hot encoded (`small` as reference/baseline category). We use
`statsmodels` for the fit because it directly reports coefficients, p-values, R², and confidence intervals,
which we need to judge statistical significance.


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("Fehlerhaft_Fahrleistung ~ days + fuel + C(engine, Treatment(reference='small'))", data=fehleranalyse).fit()
print(model.summary())


### 5.4 Checking Model Assumptions

Before trusting the coefficients, we check the key linear-regression assumptions: linearity/homoscedasticity
(residuals vs. fitted values) and approximate normality of residuals, plus multicollinearity via the
Variance Inflation Factor (VIF).


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_design = model.model.exog
vif_data = pd.DataFrame({
    "variable": model.model.exog_names,
    "VIF": [variance_inflation_factor(X_design, i) for i in range(X_design.shape[1])]
})
print(vif_data)


In [ ]:
residuals = model.resid
fitted = model.fittedvalues

fig = make_subplots(rows=1, cols=2, subplot_titles=("Residuals vs. Fitted Values", "Distribution of Residuals"))
sample_idx = np.random.choice(len(residuals), 5000, replace=False)
fig.add_trace(go.Scatter(x=fitted.iloc[sample_idx], y=residuals.iloc[sample_idx], mode="markers",
                          marker=dict(size=3, opacity=0.4), name="Residuals"), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_trace(go.Histogram(x=residuals, nbinsx=60, name="Residuals"), row=1, col=2)

fig.update_layout(height=420, width=950, showlegend=False, template="plotly_white",
                   title_text="Residual Diagnostics")
fig.show()


**Interpretation of diagnostics:**
- All **VIF values are well below the common threshold of 5–10**, so despite `fuel` and `engine` being
  related in *raw* group means, they do not cause problematic multicollinearity once both are in the model
  together as separate linear/categorical effects.
- The residuals vs. fitted plot shows a roughly random scatter around zero without a strong funnel shape,
  supporting the linearity/homoscedasticity assumption reasonably well, though some spread increases at
  higher fitted values, which is worth noting as a limitation.
- The residual distribution is approximately bell-shaped, supporting the normality assumption sufficiently
  for inference on coefficients at this sample size.


### 5.5 Interpretation and Recommendations for OEM1

**Model fit:** the model explains a substantial share of the variance in mileage at failure (see R² in the
summary above), driven almost entirely by `fuel` and `engine`, not `days`.

**Key findings:**
1. **Fuel consumption is the strongest driver of mileage at failure.** Each additional liter/100km of
   consumption is associated with several thousand additional kilometers driven before a defect occurs
   (see the `fuel` coefficient) — i.e., vehicles that consume more (typically larger/more powerful vehicles)
   are driven substantially further before failing.
2. **Engine class matters beyond fuel consumption.** Even after controlling for fuel consumption, engine
   class has an additional, statistically significant effect on mileage at failure.
3. **Elapsed time (`days`) has virtually no explanatory power.** This means failures are **not simply a
   function of vehicle age/calendar time**, but of usage intensity (approximated here by fuel consumption
   and engine class).

**Recommendations for OEM1:**
- **Prefer mileage-based over purely time-based maintenance/warranty triggers.** Since `days` barely explains
  wear, a fixed-calendar-time warranty or inspection interval will systematically under-serve
  high-consumption/high-mileage vehicles and over-serve low-mileage ones.
- **Differentiate reliability targets and inspection intervals by engine class**, since large/medium-engine
  vehicles reliably accumulate far more kilometers before failure — a uniform policy across all engine
  classes is inefficient.
- **Use fuel consumption as a low-cost proxy for usage intensity** in predictive-maintenance or
  early-warning systems, since it is strongly associated with mileage accumulation and is already recorded
  for every vehicle.
- Given the moderate (not perfect) R², other unobserved factors (driving style, road conditions, maintenance
  history) likely also matter — OEM1 should consider enriching the dataset with such variables for a more
  complete model in future analyses.


## 6. Hit and Run Accident Investigation (5 Points)

**Task:** On 11.08.2010, a hit-and-run accident occurred. The vehicle's license plate is unknown, but the
**body part number (Karosserie ID) `K5-112-1122-79`** was recovered. We need to trace which vehicle this
body belongs to and find out where it was registered.

**Approach:**
1. Find `K5-112-1122-79` in the `ID_Karosserie` column of the vehicle BOM table (`fahrzeuge_bom`) — the
   `K5` prefix already tells us it belongs to the **OEM1 Typ12** BOM file.
2. Retrieve the corresponding `ID_Fahrzeug`.
3. Look up that vehicle in `Zulassungen_alle_Fahrzeuge` to find its registration location and date.
4. Sanity-check that the registration date is before the accident date (11.08.2010), i.e. the vehicle was
   already on the road at the time of the accident.


In [ ]:
target_karosserie = "K5-112-1122-79"

match = fahrzeuge_bom[fahrzeuge_bom["ID_Karosserie"] == target_karosserie]
print("BOM record for the recovered body part:")
print(match)

vehicle_id = match["ID_Fahrzeug"].iloc[0]
print(f"\nAssociated vehicle ID: {vehicle_id}")


In [ ]:
registration = zulassungen[zulassungen["IDNummer"] == vehicle_id]
print("Registration record:")
print(registration)

accident_date = pd.Timestamp("2010-08-11")
reg_date = registration["Zulassung"].iloc[0]
print(f"\nRegistration date: {reg_date.date()}  |  Accident date: {accident_date.date()}")
print("Registered before the accident:", reg_date < accident_date)


### Result

The body part `K5-112-1122-79` belongs to vehicle **`12-1-12-82`**, which was registered in
**Aschersleben** on **2009-01-02** — well before the accident date (11.08.2010), so the registration is
consistent with this vehicle being on the road at the time of the hit-and-run. The Federal Motor Transport
Authority can direct the police to the owner of record for vehicle `12-1-12-82` in Aschersleben.
